<a href="https://colab.research.google.com/github/laxitdange28-tech/Pandas-notes/blob/main/netflix_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Netflix Movies Dataset — Data Cleaning Project
## Complete Solution Notebook

---

**Topic:** Real-World Data Cleaning with NumPy and Pandas  
**Dataset:** `netflix_movies_dirty (1).csv`

---

### What you will learn in this notebook

By the end of this project you will be able to:

1. Load a messy, real-world dataset and understand its shape and structure
2. Identify every type of data quality problem a dataset can have
3. Write Python functions that clean data step by step
4. Apply those functions to a full DataFrame using `.apply()`
5. Detect and handle missing values, duplicates, wrong data types, and outliers
6. Produce a clean, analysis-ready dataset

---

> **How to read this notebook:** Every section follows this pattern:
> - **Problem Found** — what we discovered
> - **Why is this a problem?** — the real-world impact
> - **How can we solve it?** — the plan before we code
> - Code — beginner-friendly solution
> - Explanation — what just happened

Let's begin.

---
# Step 1 — Import Libraries and Load the Dataset

Before we do anything, we import the libraries we will use throughout the project.


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("netflix_movies_dirty.csv")
df.head()

,Movie_ID,Title,Genre,Release_Year,Duration,Rating,IMDb_Rating,Votes,Director,Country,Language,Budget,Revenue,Date_Added,Age_Rating,Cast,Production_House
0,NF2602,Little Women!!!,animation,1996,3h 13m,4.7,9.5,317073,Patty Jenkins,Japan,Hin,-500000,1505957457,2015-10-14,R,"Leonardo DiCaprio, Jodie Foster",Pixar Animation Studios
1,NF3430,The White Tiger,"Animation, Drama",2007,1h 27m,8.4,4.0,123876,Jane Campion,U.S.,Kor,210868644,"687,371,362",20220620,R,"Sandra Bullock, Robert De Niro, Ryan Reynolds,...",Netflix Originals
2,NF5757,His House (Extended Collector's Edition with B...,ROMANCE,1993,1h 57m,7.4,3.4,"1,558,685",Spike Lee,India,korean,100147805,478302873,20180120,G,"Robert Downey Jr., Meryl Streep, Michael B. Jo...",Paramount Pictures
3,NF8495,Roma,"Thriller, Drama",2004,160mins,7.6,7.0,"46,529",NaN,France,KOREAN,95587293,1295144836,20210103,G,NaN,New Line Cinema
4,NF7790,Fear Street Part Three 1666,Romance,2016,113 min,5.0,5.7,682718,NaN,South Korea,French,8921122,687779882,2021-12-08,PG-13,"Timothée Chalamet, Oscar Isaac, Cate Blanchett...",Lionsgate


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1560 entries, 0 to 1559
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1560 non-null   object 
 1   Title             1483 non-null   object 
 2   Genre             1560 non-null   object 
 3   Release_Year      1560 non-null   int64  
 4   Duration          1383 non-null   object 
 5   Rating            1434 non-null   float64
 6   IMDb_Rating       1394 non-null   float64
 7   Votes             1480 non-null   object 
 8   Director          1488 non-null   object 
 9   Country           1560 non-null   object 
 10  Language          1560 non-null   object 
 11  Budget            1438 non-null   object 
 12  Revenue           1405 non-null   object 
 13  Date_Added        1560 non-null   object 
 14  Age_Rating        1560 non-null   object 
 15  Cast              1458 non-null   object 
 16  Production_House  1486 non-null   object 


---
# Step 2 — First Look at the Dataset

A real data analyst never jumps straight into cleaning. First, we look at what we have.


In [ ]:
df.shape

(1560, 17)

In [ ]:
print(f"No. of Rows = {df.shape[0]}") # row
print(f"No. of column = {df.shape[1]}") # column

No. of Rows = 1560
No. of column = 17


In [ ]:
df.columns

Index(['Movie_ID', 'Title', 'Genre', 'Release_Year', 'Duration', 'Rating',
       'IMDb_Rating', 'Votes', 'Director', 'Country', 'Language', 'Budget',
       'Revenue', 'Date_Added', 'Age_Rating', 'Cast', 'Production_House'],
      dtype='object')

In [ ]:
for i in df.columns:
  print(i)

Movie_ID
Title
Genre
Release_Year
Duration
Rating
IMDb_Rating
Votes
Director
Country
Language
Budget
Revenue
Date_Added
Age_Rating
Cast
Production_House


Notice that **every column shows `object` (string) as its dtype** — that is expected, because we loaded everything as text. Our job is to fix the types after we clean the values.

Now let's look for problems systematically.


---
# Step 3 — Finding Missing Values

## **Problem Found**

Some cells in the dataset appear empty. We need to find exactly how many are missing and in which columns.

## **Why is this a problem?**

Missing values cause errors when you try to do calculations or analysis. For example, you cannot compute the average IMDb rating if some values are empty.

## **How can we solve it?**

We use `.isnull().sum()` to count missing values column by column. We also check for **empty strings**, because sometimes a cell looks empty but actually contains `""` — which pandas does not consider as `NaN`.

In [ ]:
df.isnull().sum()

,0
Movie_ID,0
Title,77
Genre,0
Release_Year,0
Duration,177
Rating,126
IMDb_Rating,166
Votes,80
Director,72
Country,0


We can see that many columns have missing or empty values. We will deal with each one when we reach that column.

For now, let's continue discovering other problems first.


---
# Step 4 — Checking for Duplicate Movie IDs

## **Problem Found**

Each movie should have a **unique ID**. Let's check if any Movie_IDs appear more than once.

## **Why is this a problem?**

If two different rows share the same ID, we cannot tell them apart. Any analysis that relies on Movie_ID (like joining tables) will produce wrong results.

## **How can we solve it?**

We count how many unique IDs we have and compare it to the total number of rows. We also find which IDs appear more than once.

In [ ]:
# Tatal row vs unique move_ID
duplicat_ids = df['Movie_ID'].nunique()
duplicat_ids

1316

In [ ]:
# finde which moive_id apptras more then once
id_conts = df['Movie_ID'].value_counts()
id_conts

,count
Movie_ID,
NF2666,4
NF5819,4
NF6886,4
NF9963,3
NF7033,3
...,...
NF7133,1
NF7261,1
NF3377,1


In [ ]:
id_conts[id_conts > 1]

,count
Movie_ID,
NF2666,4
NF5819,4
NF6886,4
NF9963,3
NF7033,3
...,...
NF8903,2
NF9140,2
NF9252,2


## 💻 Solution — Remove Rows with Duplicate Movie IDs

We will keep the **first** occurrence of each Movie_ID and remove the rest.

> **Note:** In a real job, you might investigate each duplicate carefully. Here, we take the safe approach of keeping the first record.


In [ ]:
df = df.drop_duplicates("Movie_ID",keep="first")

In [ ]:
print(len(df))

1316


✅ **Explanation:** `drop_duplicates(subset="Movie_ID", keep="first")` looks at the Movie_ID column only and removes every row that has the same ID as a row that appeared earlier. The first occurrence is kept.


## Solution — Remove Rows with Duplicate Movie IDs

We will keep the **first** occurrence of each Movie_ID and remove the rest.

> **Note:** In a real job, you might investigate each duplicate carefully. Here, we take the safe approach of keeping the first record.

Now let's write a function to clean each title:


## Handling Rows with Missing Titles

## **Why is this a problem?**

A movie with no title is useless in the dataset — we cannot identify it. We should remove those rows.


✅ **Explanation:** We first clean every title (removing whitespace, newlines, etc.). Any title that becomes empty after cleaning is set to `NaN`. Then we drop those rows entirely because a movie without a name cannot be analysed.


---
# Step 5 — Cleaning the Title Column

## **Problem Found**

The `Title` column has many problems:

1. **Empty titles** — some rows have no title at all
2. **Extra whitespace** — leading spaces, trailing spaces, double spaces
3. **Newline characters** (`\n`) — titles end with invisible line breaks
4. **ALL CAPS titles** — some titles are fully uppercased
5. **Exclamation marks** — titles like `"Inception!!!"`
6. **Excessively long titles** — titles like `"His House (Extended Collector's Edition with Bonus Features and Director's Commentary)"`

## **Why is this a problem?**

If two rows have the title `"Inception"` and `"  inception\n"`, they look like different movies to a computer even though they are the same.

## **How can we solve it?**

We will write a function that handles each problem one by one. Then we apply that function to every title in the dataset.



Now let's write a function to clean each title:



In [ ]:
""" Task:
1. handale missing/NaA
2. Remove /n or /t
3. Remove extra whitespace
4. Remove punctuation like!!!
5. Truncate long titles
6. Convert to title case
"""
def clean_title(title):

  if pd.isnull(title):
    return np.nan

  title = title.replace("\n"," ")
  title = title.replace("\t"," ")

  title = " ".join(title.split())

  title  = title.rstrip('!')

  if '(' in title:
    title = title[:title.index('(')].strip()

  title = title.title()

  return title


In [ ]:
df['Title'] = df['Title'].apply(clean_title)

In [ ]:
df['Title']


,Title
0,Little Women
1,The White Tiger
2,His House
3,Roma
4,Fear Street Part Three 1666
...,...
1555,The Hand Of God
1556,Mass
1557,NaN
1558,Demon Slayer Mugen Train


✅ **Explanation:** The key idea is to pick **one separator** and split on it. We convert all separators (`,`, `|`, `/`) into `|`, then take only the part before the first `|`. This gives us the primary genre. Title Case makes every genre consistent.


---
# Step 6 — Standardising the Genre Column

## **Problem Found**

The `Genre` column has many inconsistencies:

- `"action"`, `"ACTION"`, `"Action"` — same genre, different casing
- `"Action "`, `" Fantasy"`, `"\tDrama"` — leading/trailing spaces and tab characters
- `"Action, Drama"`, `"Action | Drama"`, `"Drama/Comedy"` — multiple genres joined with different separators

## **Why is this a problem?**

If you try to count how many action movies there are, `"action"`, `"ACTION"`, and `"Action "` will all be counted separately. You will get the wrong answer.

## **How can we solve it?**

We will extract only the **first genre** from each row (the primary genre), then standardise its casing and remove extra whitespace. This gives us one clean category per movie.

In [ ]:
df['Genre']

,Genre
0,animation
1,"Animation, Drama"
2,ROMANCE
3,"Thriller, Drama"
4,Romance
...,...
1555,Drama
1556,Thriller
1557,THRILLER
1558,ACTION


In [ ]:
def clean_genre(genre):
  if pd.isnull(genre):
    return np.nan

  genre =  genre.replace("\n"," ")
  genre =  genre.replace("\t"," ")

  genre = "".join(genre.split())
  genre = genre.split(',')[0]
  genre = genre.title()

  return genre


In [ ]:
df['Genre'] = df['Genre'].apply(clean_genre)

In [ ]:
genre = "Animation,Drama"
genre.split(',')[0]

'Animation'

In [ ]:
df['Genre']

,Genre
0,Animation
1,Animation
2,Romance
3,Thriller
4,Romance
...,...
1555,Drama
1556,Thriller
1557,Thriller
1558,Action


✅ **Explanation:** We set a sensible boundary: 1900 to 2025. Any value outside that range is treated as an error and replaced with `NaN`. We do not guess the correct year because we have no way of knowing what it should be.


---
# Step 7 — Fixing the Release Year Column

## **Problem Found**

The `Release_Year` column contains **outlier values** that are impossible or unrealistic:

- `1890` — movies did not exist in 1890
- `2099` — that is in the future

## **Why is this a problem?**

If we calculate the average release year, impossible values will pull the result in the wrong direction. We also cannot correctly sort movies by year.

## **How can we solve it?**

We convert the column to numbers. Then we set any year outside a realistic range (1900–2025) to `NaN` (missing), so they are excluded from analysis.

In [ ]:
df['Release_Year']

,Release_Year
0,1996
1,2007
2,1993
3,2004
4,2016
...,...
1555,2017
1556,1996
1557,2016
1558,2017


In [ ]:
def clean_year(year):
  if pd.isnull(year):
    return np.nan

  if year <1900 or year >2025:
    return np.nan

  return year

In [ ]:
df['Release_Year'] = df['Release_Year'].apply(clean_year)
df

,Movie_ID,Title,Genre,Release_Year,Duration,Rating,IMDb_Rating,Votes,Director,Country,Language,Budget,Revenue,Date_Added,Age_Rating,Cast,Production_House
0,NF2602,Little Women,Animation,1996.0,3h 13m,4.7,9.5,317073,Patty Jenkins,Japan,Hin,-500000,1505957457,2015-10-14,R,"Leonardo DiCaprio, Jodie Foster",Pixar Animation Studios
1,NF3430,The White Tiger,Animation,2007.0,1h 27m,8.4,4.0,123876,Jane Campion,U.S.,Kor,210868644,"687,371,362",20220620,R,"Sandra Bullock, Robert De Niro, Ryan Reynolds,...",Netflix Originals
2,NF5757,His House,Romance,1993.0,1h 57m,7.4,3.4,"1,558,685",Spike Lee,India,korean,100147805,478302873,20180120,G,"Robert Downey Jr., Meryl Streep, Michael B. Jo...",Paramount Pictures
3,NF8495,Roma,Thriller,2004.0,160mins,7.6,7.0,"46,529",NaN,France,KOREAN,95587293,1295144836,20210103,G,NaN,New Line Cinema
4,NF7790,Fear Street Part Three 1666,Romance,2016.0,113 min,5.0,5.7,682718,NaN,South Korea,French,8921122,687779882,2021-12-08,PG-13,"Timothée Chalamet, Oscar Isaac, Cate Blanchett...",Lionsgate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1555,NF4901,The Hand Of God,Drama,2017.0,135mins,8.3,5.0,1525690,James Gunn,France,KOREAN,NaN,"730,299,883",28/10/2015,NR,"Lupita Nyong'o, Margot Robbie, Pedro Pascal, L...",DC Films
1556,NF4202,Mass,Thriller,1996.0,121 min,6.4,5.8,1779958,Spike Lee,U.K.,ENGLISH,$61305141,42703073,"November 26, 2022",R,"Pedro Pascal, Brad Pitt, Al Pacino",Apple TV+
1557,NF9751,NaN,Thriller,2016.0,1h 43m,6.8,7.1,768218,Francis Ford Coppola,Italy,ENGLISH,"$80,210,058","1,724,230,587","May 19, 2018",NC-17,"Al Pacino, Angelina Jolie",Marvel Studios
1558,NF4195,Demon Slayer Mugen Train,Action,2017.0,NaN,4.5,-1.0,141444,Ryan Coogler,South Korea,FRENCH,202601908,NaN,12/11/2021,PG-13,"Scarlett Johansson, Lupita Nyong'o",Columbia Pictures


✅ **Explanation:** The key insight is that we handle each format with a separate `if-elif` block. The `"h"` character is our detector for the hours-and-minutes format. For everything else, we extract only the digits. The outlier filter (30–600 minutes) removes impossible values.


---
# Step 9 — Converting the Rating Column

## **Problem Found**

The `Rating` column contains numbers but they are **stored as text (strings)** because we loaded the file with `dtype=str`. Some values are also missing.

## **Why is this a problem?**

You cannot do arithmetic on text. `"8.5" + "7.0"` is `"8.57.0"` in Python, not `15.5`.

## **How can we solve it?**

We use `pd.to_numeric()` to convert the column to floating-point numbers. The `errors="coerce"` argument automatically turns any non-numeric value into `NaN`.

✅ **Explanation:** `pd.to_numeric(errors="coerce")` is the cleanest way to convert a text column to numbers when you know some values might not be valid numbers. Any value that cannot become a number silently becomes `NaN`.


---
# Step 10 — Cleaning the IMDb_Rating Column

## **Problem Found**

The IMDb rating scale runs from **0 to 10**. Our dataset contains values outside this range:

- Values **greater than 10** (e.g. `15.0`) — impossible on IMDb
- Values **below 0** (e.g. `-1.0`) — also impossible

## **Why is this a problem?**

These outliers will corrupt any analysis involving IMDb ratings. The maximum, average, and sorting will all be wrong.

## **How can we solve it?**

1. Convert the column to floats
2. Replace any value outside `[0, 10]` with `NaN`

In [ ]:
upper = (df['IMDb_Rating'] >10).sum()
upper

np.int64(45)

In [ ]:
lower = (df['IMDb_Rating'] <0).sum()
lower

np.int64(43)

In [ ]:
def clean_IMDb(rating):
  if pd.isnull(rating):
    return np.nan

  if rating >10 or rating <0:
    return np.nan

  return rating


In [ ]:
df['IMDb_Rating'] = df['IMDb_Rating'].apply(clean_IMDb)
df['IMDb_Rating']

,IMDb_Rating
0,9.5
1,4.0
2,3.4
3,7.0
4,5.7
...,...
1555,5.0
1556,5.8
1557,7.1
1558,NaN


✅ **Explanation:** The IMDb rating range (0–10) is a known domain rule. Any value outside that range is a data entry error, so we set it to `NaN`. This is called **domain-rule validation**.


---
# Step 11 — Cleaning the Votes Column

## **Problem Found**

The `Votes` column contains numbers, but some are formatted with **commas** (e.g. `"1,558,685"`). Pandas treats these as text, not numbers.

## **Why is this a problem?**

`"1,558,685"` is a string. You cannot sort or sum strings that look like numbers.

## **How can we solve it?**

We write a function that removes the commas and converts the value to an integer.

In [ ]:
df

,Movie_ID,Title,Genre,Release_Year,Duration,Rating,IMDb_Rating,Votes,Director,Country,Language,Budget,Revenue,Date_Added,Age_Rating,Cast,Production_House
0,NF2602,Little Women,Animation,1996.0,3h 13m,4.7,9.5,317073,Patty Jenkins,Japan,Hin,-500000,1505957457,2015-10-14,R,"Leonardo DiCaprio, Jodie Foster",Pixar Animation Studios
1,NF3430,The White Tiger,Animation,2007.0,1h 27m,8.4,4.0,123876,Jane Campion,U.S.,Kor,210868644,"687,371,362",20220620,R,"Sandra Bullock, Robert De Niro, Ryan Reynolds,...",Netflix Originals
2,NF5757,His House,Romance,1993.0,1h 57m,7.4,3.4,"1,558,685",Spike Lee,India,korean,100147805,478302873,20180120,G,"Robert Downey Jr., Meryl Streep, Michael B. Jo...",Paramount Pictures
3,NF8495,Roma,Thriller,2004.0,160mins,7.6,7.0,"46,529",NaN,France,KOREAN,95587293,1295144836,20210103,G,NaN,New Line Cinema
4,NF7790,Fear Street Part Three 1666,Romance,2016.0,113 min,5.0,5.7,682718,NaN,South Korea,French,8921122,687779882,2021-12-08,PG-13,"Timothée Chalamet, Oscar Isaac, Cate Blanchett...",Lionsgate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1555,NF4901,The Hand Of God,Drama,2017.0,135mins,8.3,5.0,1525690,James Gunn,France,KOREAN,NaN,"730,299,883",28/10/2015,NR,"Lupita Nyong'o, Margot Robbie, Pedro Pascal, L...",DC Films
1556,NF4202,Mass,Thriller,1996.0,121 min,6.4,5.8,1779958,Spike Lee,U.K.,ENGLISH,$61305141,42703073,"November 26, 2022",R,"Pedro Pascal, Brad Pitt, Al Pacino",Apple TV+
1557,NF9751,NaN,Thriller,2016.0,1h 43m,6.8,7.1,768218,Francis Ford Coppola,Italy,ENGLISH,"$80,210,058","1,724,230,587","May 19, 2018",NC-17,"Al Pacino, Angelina Jolie",Marvel Studios
1558,NF4195,Demon Slayer Mugen Train,Action,2017.0,NaN,4.5,NaN,141444,Ryan Coogler,South Korea,FRENCH,202601908,NaN,12/11/2021,PG-13,"Scarlett Johansson, Lupita Nyong'o",Columbia Pictures


In [ ]:
# We write a function that removes the commas and converts the value to an integer.
# def clean_votes(votes):
#   if pd.isnull(votes):
#     return np.nan

#   votes = votes.replace(",","")

#   if votes.isdigit():
#     return int(votes)

#   return np.nan

In [ ]:
a = "1,558,685"
aa = "".join(a.split(','))
aa

'1558685'

In [ ]:
def clean_votes(votes):
  if pd.isnull(votes):
    return np.nan

  votes = "".join(votes.split(','))
  return int(votes)



In [ ]:
df['Votes'] = df['Votes'].apply(clean_votes)
df['Votes']

,Votes
0,317073.0
1,123876.0
2,1558685.0
3,46529.0
4,682718.0
...,...
1555,1525690.0
1556,1779958.0
1557,768218.0
1558,141444.0


✅ **Explanation:** The `.replace(",", "")` removes the comma separators. Then `.isdigit()` confirms the remaining characters are all digits before we convert to `int`.


---
# Step 12 — Cleaning the Budget Column

## **Problem Found**

The `Budget` column has two problems:

1. **Dollar signs and commas** — e.g. `"$176,890,118"` is text, not a number
2. **Negative values** — e.g. `"-500000"` — a budget cannot be negative

## **Why is this a problem?**

We cannot do any financial analysis (like comparing budget to revenue) until budget is a clean positive number.

## **How can we solve it?**

Remove the `$` and `,`, convert to a number, then set negative values to `NaN`.

In [ ]:
df['Budget']

,Budget
0,-500000
1,210868644
2,100147805
3,95587293
4,8921122
...,...
1555,NaN
1556,$61305141
1557,"$80,210,058"
1558,202601908


In [ ]:
def clean_budget(budget):
  """
  1. Remove $ and,
  2. Convert to float
  3. Set negative values to NaN
  """
  if pd.isnull(budget):
    return np.nan

  budget = budget.replace("$","")
  budget = budget.replace(",","")

  budget = float(budget)

  if budget < 0:
    return np.nan

  return budget


In [ ]:
df['Budget'] = df['Budget'].apply(clean_budget)
df['Budget']

,Budget
0,NaN
1,210868644.0
2,100147805.0
3,95587293.0
4,8921122.0
...,...
1555,NaN
1556,61305141.0
1557,80210058.0
1558,202601908.0


✅ **Explanation:** We use a `try-except` block because after removing `$` and `,`, there might still be values that cannot be converted (like stray text). The `try-except` catches those safely and returns `NaN`. Then we apply the domain rule: budget ≥ 0.


---
# Step 13 — Cleaning the Revenue Column

## **Problem Found**

The `Revenue` column has:

1. **Commas in numbers** — e.g. `"687,371,362"`
2. **Revenue = 0** — which is unrealistic for a movie that made it onto Netflix

## **Why is this a problem?**

Revenue of 0 is almost certainly a data entry error. A movie that generated no money whatsoever is not meaningful in most analyses.

## **How can we solve it?**

Remove commas, convert to float, then set revenue <= 0 to `NaN`.

In [ ]:
df['Revenue']

,Revenue
0,1505957457
1,"687,371,362"
2,478302873
3,1295144836
4,687779882
...,...
1555,"730,299,883"
1556,42703073
1557,"1,724,230,587"
1558,NaN


In [ ]:
def clean_revenue(Revenue):
  """
  1. Remove commas
  2. Convert to float
  3. Set 0 or revenue to NaN
  """

  if pd.isnull(Revenue):
    return np.nan

  Revenue = Revenue.replace(",","")
  Revenue = float(Revenue)

  if Revenue <=0:
    return np.nan

  return Revenue

In [ ]:
df['Revenue'] = df['Revenue'].apply(clean_revenue)
df['Revenue']

,Revenue
0,1.505957e+09
1,6.873714e+08
2,4.783029e+08
3,1.295145e+09
4,6.877799e+08
...,...
1555,7.302999e+08
1556,4.270307e+07
1557,1.724231e+09
1558,NaN


✅ **Explanation:** The logic is the same as Budget, with one addition: revenue = 0 is also an outlier. We use `<= 0` instead of `< 0` to catch both zero and negative values.


---
# Step 14 — Standardising the Date_Added Column

## **Problem Found**

The `Date_Added` column stores dates in at least **four different formats**:

- `"2024-05-01"` — ISO format (Year-Month-Day)
- `"01/05/2024"` — Day/Month/Year
- `"May 1, 2024"` — Month name Day, Year
- `"20240501"` — Compact format (no separators)

## **Why is this a problem?**

You cannot sort or compare dates stored as text in different formats. `"20230101"` and `"January 1, 2023"` are the same date but Python cannot know that.

## **How can we solve it?**

We use `pd.to_datetime()` with `infer_datetime_format=True`, which is smart enough to recognise most formats automatically. For the compact format (`"20230101"`), we add a special pre-processing step.

In [ ]:
df['Date_Added']

,Date_Added
0,2015-10-14
1,20220620
2,20180120
3,20210103
4,2021-12-08
...,...
1555,28/10/2015
1556,"November 26, 2022"
1557,"May 19, 2018"
1558,12/11/2021


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1316 entries, 0 to 1559
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1316 non-null   object 
 1   Title             1251 non-null   object 
 2   Genre             1316 non-null   object 
 3   Release_Year      1233 non-null   float64
 4   Duration          1164 non-null   object 
 5   Rating            1212 non-null   float64
 6   IMDb_Rating       1092 non-null   float64
 7   Votes             1247 non-null   float64
 8   Director          1255 non-null   object 
 9   Country           1316 non-null   object 
 10  Language          1316 non-null   object 
 11  Budget            1170 non-null   float64
 12  Revenue           1144 non-null   float64
 13  Date_Added        1316 non-null   object 
 14  Age_Rating        1316 non-null   object 
 15  Cast              1235 non-null   object 
 16  Production_House  1252 non-null   object 
dtype

In [ ]:
def clean_date(date):
  if pd.isnull(date):
    return np.nan

  # date = pd.to_datetime(date,infer_datetime_format=True)
  date = pd.to_datetime(df['Date_Added'],format='mixed')

  return date

In [ ]:
df['Date_Added'] = pd.to_datetime(df['Date_Added'],format='mixed',errors='coerce').dt.strftime("%d-%m-%Y")


In [ ]:
df['Date_Added']

,Date_Added
0,14-10-2015
1,20-06-2022
2,20-01-2018
3,03-01-2021
4,08-12-2021
...,...
1555,28-10-2015
1556,26-11-2022
1557,19-05-2018
1558,11-12-2021


✅ **Explanation:** The compact format (`"20240501"`) is the tricky one — pandas cannot detect it automatically. We handle it by checking if the string is 8 digits long, then inserting dashes to convert it to `"2024-05-01"`. After that, `pd.to_datetime()` handles everything else.


---
# Step 15 — Standardising the Country Column

## **Problem Found**

The same countries appear with different names:

- `"USA"`, `"United States"`, `"U.S."`, `"US"`, `"U.S.A"` — all mean the United States
- `"UK"`, `"United Kingdom"`, `"Britain"`, `"U.K."` — all mean the United Kingdom

## **Why is this a problem?**

If we group movies by country, the USA will appear as 5 separate groups instead of one. We will grossly undercount American movies.

## **How can we solve it?**

We build a **dictionary** that maps every known variant to the official name. Then we write a function that looks up each value in the dictionary.

In [ ]:
df['Country'].unique()

array(['Japan', 'U.S.', 'India', 'France', 'South Korea', 'Germany',
       'Britain', 'Italy', 'US', 'U.K.', 'USA', 'U.S.A', 'United Kingdom',
       'United States', 'UK', 'usa'], dtype=object)

In [ ]:
#Formet united State
# def clean_country(country):
#   if pd.isnull(country):
#     return np.nan

#   if country == "USA":
#     return "United States"
#   elif country == "United States":
#     return "United States"
#   elif country == "U.S.":
#     return "United States"
#   elif country == "US":
#     return "United States"
#   elif country == "U.S.A":
#     return "United States"



In [ ]:
# another step use dictionary
def clean_country(country):
  if pd.isnull(country):
    return np.nan

  country_dict = {
      "USA": "United States",
      "United States": "United States",
      "U.S.": "United States",
      "US": "United States",
      "U.S.A": "United States",
      "UK": "United Kingdom",
      "United Kingdom": "United Kingdom",
      "Britain": "United Kingdom",
      "U.K.": "United Kingdom"
  }

  if country in country_dict:
    return country_dict[country]
  else:
    return country


In [ ]:
df['Country'] = df['Country'].apply(clean_country)
df['Country']

,Country
0,Japan
1,United States
2,India
3,France
4,South Korea
...,...
1555,France
1556,United Kingdom
1557,Italy
1558,South Korea


✅ **Explanation:** A dictionary is the perfect tool for this problem. We store every known variant as a key (all lowercase) and the correct name as the value. Converting the input to lowercase before lookup means we handle `"USA"`, `"usa"`, and `"Usa"` all the same way.


---
# Step 16 — Standardising the Language Column

## **Problem Found**

The `Language` column has:

- Inconsistent casing: `"english"`, `"ENGLISH"`, `"English"`
- Abbreviations: `"Eng"`, `"Hin"`, `"Kor"`, `"Ger"`, `"Fre"`, `"Jap"`, `"Spa"`, `"Ita"`

## **Why is this a problem?**

`"english"` and `"ENGLISH"` will be counted as different languages. Abbreviations like `"Eng"` might not even be recognised in reports.

## **How can we solve it?**

Build a dictionary mapping abbreviations and incorrect casings to the full, properly-capitalised language name.


In [ ]:
df['Language'].unique()

array(['Hin', 'Kor', 'korean', 'KOREAN', 'French', 'Ger', 'English',
       'Spanish', 'Italian', 'japanese', 'JAPANESE', 'Japanese', 'french',
       'German', 'italian', 'HINDI', 'FRENCH', 'Eng', 'Spa', 'Hindi',
       'english', 'GERMAN', 'german', 'Fre', 'spanish', 'Korean', 'Jap',
       'ITALIAN', 'Ita', 'SPANISH', 'ENGLISH', 'hindi'], dtype=object)

In [ ]:
def clean_language(language):
  if pd.isnull(language):
    return np.nan

  lang_dict = {
      "English": "English",
      "english": "English",
      "ENGLISH": "English",
      "Eng": "English",
      "Hin": "Hindi",
      "Kor": "Korean",
      "Ger": "German",
      "Fre": "French",
      "Jap": "Japanese",
      "Spa": "Spanish",
      "Ita": "Italian"

  }

  if language in lang_dict:
    return lang_dict[language]
  else:
    return language.title()

In [ ]:
df['Language'] = df['Language'].apply(clean_language)
df['Language']

,Language
0,Hindi
1,Korean
2,Korean
3,Korean
4,French
...,...
1555,Korean
1556,English
1557,English
1558,French


✅ **Explanation:** The same dictionary-lookup pattern we used for Country works perfectly here. All abbreviations like `"Hin"` become `"Hindi"` and all casing variants like `"ENGLISH"` become `"English"`.


---
# Step 16 — Standardising the Language Column

## **Problem Found**

The `Language` column has:

- Inconsistent casing: `"english"`, `"ENGLISH"`, `"English"`
- Abbreviations: `"Eng"`, `"Hin"`, `"Kor"`, `"Ger"`, `"Fre"`, `"Jap"`, `"Spa"`, `"Ita"`

## **Why is this a problem?**

`"english"` and `"ENGLISH"` will be counted as different languages. Abbreviations like `"Eng"` might not even be recognised in reports.

## **How can we solve it?**

Build a dictionary mapping abbreviations and incorrect casings to the full, properly-capitalised language name.

✅ **Explanation:** We reuse the same simple function for all three columns. It strips whitespace and converts empty strings to `NaN`, so all missing values are represented the same way.


---
# Step 18 — Final Duplicate Check (Near-Duplicate Rows)

## **Problem Found**

Even after removing duplicate Movie_IDs, the dataset may still contain rows that are very similar — the same title and year but with slightly different other values.

## **Why is this a problem?**

If the same movie appears twice with slightly different data, it will be double-counted in analysis.

## **How can we solve it?**

We check for rows where both `Title` AND `Release_Year` are the same. For any such group, we keep only the first row.

In [ ]:
df[df.duplicated(subset=["Title", "Release_Year"])]



,Movie_ID,Title,Genre,Release_Year,Duration,Rating,IMDb_Rating,Votes,Director,Country,Language,Budget,Revenue,Date_Added,Age_Rating,Cast,Production_House
62,NF9140,Halloween Kills,Documentary,2023.0,149mins,5.7,NaN,1454314.0,Park Chan-wook,Italy,French,104634025.0,NaN,22-06-2018,G,"Will Smith, Michael B. Jordan, Jodie Foster",Netflix Originals
140,NF1157,NaN,Adventure,2009.0,96 min,6.3,9.6,586936.0,James Cameron,Germany,Japanese,201262717.0,NaN,05-09-2016,TV-14,"Denzel Washington, Florence Pugh",Columbia Pictures
242,NF2035,Cherry,Sci-Fi,2000.0,182 min,4.7,7.3,330933.0,Martin Scorsese,United Kingdom,English,190214408.0,1.904736e+09,23-07-2019,TV-PG,"Al Pacino, Will Smith, Morgan Freeman, Margot ...",DC Films
257,NF9343,Malignant,Adventure/Comedy,1993.0,198,4.9,NaN,832816.0,J.J. Abrams,South Korea,Italian,117309547.0,2.447163e+08,05-06-2015,TV-PG,"Leonardo DiCaprio, Adam Driver, Ryan Reynolds",Focus Features
260,NF3292,The Irishman,Action,2009.0,NaN,8.1,6.2,965368.0,Darren Aronofsky,United Kingdom,French,244522167.0,1.549798e+09,01-04-2017,PG-13,"Natalie Portman, Timothée Chalamet, Margot Rob...",20th Century Studios
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530,NF9667,NaN,Fantasy,2019.0,169mins,5.5,6.1,1876871.0,Francis Ford Coppola,South Korea,German,117695199.0,1.014116e+09,13-07-2019,NR,"Cate Blanchett, Florence Pugh, Will Smith",Columbia Pictures
1542,NF6294,The Mitchells Vs The Machines,Sci-Fi,NaN,108mins,8.8,9.3,748394.0,Denis Villeneuve,Italy,Italian,240420306.0,1.724165e+09,14-11-2024,PG-13,"Dwayne Johnson, Oscar Isaac, Margot Robbie",Blumhouse Productions
1550,NF3575,NaN,Comedy,1990.0,88,5.7,NaN,599610.0,Ridley Scott,India,Spanish,178067761.0,1.508614e+09,26-04-2017,TV-MA,"Cate Blanchett, Leonardo DiCaprio",Universal Pictures
1557,NF9751,NaN,Thriller,2016.0,1h 43m,6.8,7.1,768218.0,Francis Ford Coppola,Italy,English,80210058.0,1.724231e+09,19-05-2018,NC-17,"Al Pacino, Angelina Jolie",Marvel Studios


In [ ]:
df.duplicated(subset=["Title", "Release_Year"]).sum()

np.int64(137)

In [ ]:
df.drop_duplicates(subset=["Title", "Release_Year"],inplace=True)

In [ ]:
df.duplicated(subset=["Title", "Release_Year"]).sum()

np.int64(0)

✅ **Explanation:** `drop_duplicates(subset=["Title", "Release_Year"])` looks at both columns together. Two rows must share both the same title AND the same year to be considered duplicates. This avoids accidentally removing movies that share a title but are from different years (like sequels or remakes).


---
# Step 19 — Filling Remaining Missing Values

## **Problem Found**

After all the cleaning above, some columns still have missing values. We need a strategy for each one.

## Strategy by column

| Column | Strategy | Reason |
|---|---|---|
| `Duration` | Fill with median | Median is not affected by outliers |
| `Rating` | Fill with median | Same reason |
| `IMDb_Rating` | Fill with median | Same reason |
| `Votes` | Fill with median | Same reason |
| `Budget` | Leave as NaN | Cannot guess budget — too important to fake |
| `Revenue` | Leave as NaN | Same |
| `Release_Year` | Leave as NaN | Wrong year is worse than no year |
| `Director`, `Cast`, `Production_House` | Leave as NaN | Cannot guess names |

In [ ]:
df.isnull().sum()

,0
Movie_ID,0
Title,32
Genre,0
Release_Year,69
Duration,137
Rating,92
IMDb_Rating,205
Votes,67
Director,54
Country,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1179 entries, 0 to 1558
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1179 non-null   object 
 1   Title             1147 non-null   object 
 2   Genre             1179 non-null   object 
 3   Release_Year      1110 non-null   float64
 4   Duration          1042 non-null   object 
 5   Rating            1087 non-null   float64
 6   IMDb_Rating       974 non-null    float64
 7   Votes             1112 non-null   float64
 8   Director          1125 non-null   object 
 9   Country           1179 non-null   object 
 10  Language          1179 non-null   object 
 11  Budget            1043 non-null   float64
 12  Revenue           1018 non-null   float64
 13  Date_Added        1179 non-null   object 
 14  Age_Rating        1179 non-null   object 
 15  Cast              1104 non-null   object 
 16  Production_House  1120 non-null   object 
dtype

In [ ]:
nume_column = df.select_dtypes(include=['int64','float64']).columns.tolist()
nume_column

['Release_Year', 'Rating', 'IMDb_Rating', 'Votes', 'Budget', 'Revenue']

In [ ]:
numeric_column = df.select_dtypes(include=['int64','float64']).skew()
numeric_column

,0
Release_Year,-0.026920
Rating,-0.042045
IMDb_Rating,0.070337
Votes,-0.056963
Budget,0.005683
Revenue,-0.022881


## Numeric column --> Meadin --> Skewness() --> mean

## Object/String --> Mode, PlaceHolder.

In [ ]:
#skewness --> median
# NO near to 0 skewness --> mean
df['Release_Year'] = df['Release_Year'].fillna(df['Release_Year'].median())
df['Release_Year']


,Release_Year
0,1996.0
1,2007.0
2,1993.0
3,2004.0
4,2016.0
...,...
1553,1993.0
1554,2019.0
1555,2017.0
1556,1996.0


In [ ]:
df.isnull().sum()

,0
Movie_ID,0
Title,32
Genre,0
Release_Year,0
Duration,137
Rating,92
IMDb_Rating,205
Votes,67
Director,54
Country,0


In [ ]:
df['IMDb_Rating'] = df['IMDb_Rating'].fillna(df['IMDb_Rating'].mean())
df['IMDb_Rating']

,IMDb_Rating
0,9.500000
1,4.000000
2,3.400000
3,7.000000
4,5.700000
...,...
1553,5.000000
1554,5.500000
1555,5.000000
1556,5.800000


In [ ]:
df.isnull().sum()

,0
Movie_ID,0
Title,32
Genre,0
Release_Year,0
Duration,137
Rating,92
IMDb_Rating,0
Votes,67
Director,54
Country,0


In [ ]:
df['Title'] = df['Title'].fillna('Unknown')
df['Title'].head(50)

,Title
0,Little Women
1,The White Tiger
2,His House
3,Roma
4,Fear Street Part Three 1666
5,Another Round
6,Outer Banks
7,Space Jam A New Legacy
8,Candyman
9,The Lost Daughter


In [ ]:
df['Rating'] = df['Rating'].fillna(df['Rating'].median())
df['Rating']

,Rating
0,4.7
1,8.4
2,7.4
3,7.6
4,5.0
...,...
1553,4.8
1554,6.2
1555,8.3
1556,6.4


In [ ]:
df['Votes'] = df['Votes'].fillna(df['Votes'].median())
df['Votes']

,Votes
0,317073.0
1,123876.0
2,1558685.0
3,46529.0
4,682718.0
...,...
1553,457565.0
1554,1328440.0
1555,1525690.0
1556,1779958.0


In [ ]:
df['Budget'] = df['Budget'].fillna(df['Budget'].mean())
df['Budget']

,Budget
0,1.220287e+08
1,2.108686e+08
2,1.001478e+08
3,9.558729e+07
4,8.921122e+06
...,...
1553,1.151518e+08
1554,2.026018e+08
1555,1.220287e+08
1556,6.130514e+07


In [ ]:
df['Revenue'] = df['Revenue'].fillna(df['Revenue'].mean())
df['Revenue']

,Revenue
0,1.505957e+09
1,6.873714e+08
2,4.783029e+08
3,1.295145e+09
4,6.877799e+08
...,...
1553,3.613560e+08
1554,1.315436e+09
1555,7.302999e+08
1556,4.270307e+07


In [ ]:
df['Duration']

,Duration
0,3h 13m
1,1h 27m
2,1h 57m
3,160mins
4,113 min
...,...
1553,147
1554,188mins
1555,135mins
1556,121 min


In [ ]:
def clean_duration(duration):
  if pd.isnull(duration):
    return np.nan

  if 'h' in duration:
    hours , minutes = duration.split("h")
    hours = int(hours)*60

    minutes = int(minutes.replace("m",''))
    return hours + minutes

  elif "min" in duration:
      duration = duration.replace("mins",'')
      duration = duration.replace("min",'')
      return int(duration)

  else:
    return int(duration)

In [ ]:
df['Duration'] = df['Duration'].apply(clean_duration)
df['Duration']

,Duration
0,193.0
1,87.0
2,117.0
3,160.0
4,113.0
...,...
1553,147.0
1554,188.0
1555,135.0
1556,121.0


✅ **Explanation:** We use `np.median()` to compute the median from a NumPy array. The median is better than the mean for filling missing values because it is not pulled by extreme values. We intentionally leave Budget, Revenue, and year-related columns as `NaN` because guessing wrong values would be worse than having no value.


---
# Step 20 — Final Data Type Conversion

## **Problem Found**

Some columns that should be integers (like `Release_Year`, `Votes`, `Duration`) are still stored as floats because of the `NaN` values we introduced during cleaning. Pandas requires integer columns to have no missing values.

## **How can we solve it?**

We use pandas **nullable integer type** (`"Int64"`) which allows integers with missing values. For columns with no missing values, we convert to standard `int`.

✅ **Explanation:** `"Int64"` (capital I) is pandas' nullable integer type. It can store whole numbers AND `NaN` at the same time, which regular Python `int` cannot do.


---
# Step 21 — Final Verification

Now that all cleaning is done, let's verify that our dataset is truly clean.


---
# Step 22 — Before vs After Comparison

Let's compare key statistics from the raw dataset to the cleaned dataset.


---
# Step 23 — Save the Cleaned Dataset

Finally, we save the cleaned dataset as a new CSV file so the original messy file is preserved.


---

# 🎉 Project Complete!

Here is a summary of everything we cleaned in this project:

| Step | Problem | Solution |
|------|---------|----------|
| 4 | Duplicate Movie_IDs | `drop_duplicates(subset="Movie_ID")` |
| 5 | Messy titles (spaces, newlines, caps, long) | Custom `clean_title()` function |
| 6 | Inconsistent genre formats | Custom `clean_genre()` + dictionary |
| 7 | Release year outliers (1890, 2099) | Domain-rule validation |
| 8 | Duration in mixed formats | Custom `convert_duration()` function |
| 9 | Rating stored as string | `pd.to_numeric()` |
| 10 | IMDb ratings outside 0–10 | Domain-rule validation |
| 11 | Votes with commas | Remove commas, convert to int |
| 12 | Budget with `$` signs and negatives | Remove `$`/`,`, check ≥ 0 |
| 13 | Revenue with commas, zero values | Remove `,`, check > 0 |
| 14 | Dates in 4 different formats | `pd.to_datetime()` + compact format handler |
| 15 | Country name variants | Lookup dictionary |
| 16 | Language abbreviations and casing | Lookup dictionary |
| 17 | Missing Director/Cast/Production House | Standardised to `NaN` |
| 18 | Near-duplicate rows (same Title + Year) | `drop_duplicates(subset=["Title","Release_Year"])` |
| 19 | Remaining missing numeric values | Fill with median |
| 20 | Wrong data types | `astype()` and `"Int64"` |

**Key Python concepts used:**

- `def` functions with clear parameters
- `for` loops and `if-elif-else` conditions
- `str.strip()`, `str.replace()`, `str.split()`, `str.lower()`, `str.title()`
- Dictionaries for lookup tables
- NumPy arrays and aggregate functions
- `pd.to_numeric()`, `pd.to_datetime()`
- `df.apply()` to run a function on every row
- `df.dropna()`, `df.fillna()`, `df.drop_duplicates()`
